# 06 - Raw Gemini Function Calling

## Learning objective

Remove LangChain tool-calling abstractions and implement the provider-specific Gemini workflow directly.

```text
Manual schema -> Gemini FunctionCall -> Python function -> Gemini FunctionResponse -> final answer
```

This notebook breaks the implementation into small, inspectable steps.

> **Quota note:** One complete shopping run normally makes three Gemini requests.


## 1. Check the active Python kernel

The executable should point to the Conda environment used for notebooks, such as `nlp_env`.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

If the course packages are not installed in the active kernel, remove the `#` and run the cell once. Then use **Kernel -> Restart Kernel**.


In [ ]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL and environment variables

`truststore` uses the Windows certificate store. The existing `.env` is loaded from the sibling script folder without printing secret values.


In [2]:
import os
from pathlib import Path

import truststore
from dotenv import load_dotenv

truststore.inject_into_ssl()

ENV_PATH = Path("../langchain-course/.env").resolve()
load_dotenv(dotenv_path=ENV_PATH)

print(".env exists:", ENV_PATH.exists())
print("Google key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))


.env exists: True
Google key loaded: True
Tavily key loaded: True
LangSmith key loaded: True
LangSmith tracing: true


## 4. Import the raw Google Gen AI SDK and tracing support


In [3]:
import json

from google import genai
from google.genai import types
from langsmith import traceable

MAX_ITERATIONS = 10
MODEL = "gemini-3.5-flash-lite" #"gemini-3.6-flash"
client = genai.Client()


## 5. Define regular Python functions

There is no LangChain `@tool`. The functions and model-visible schemas are separate.


In [4]:
@traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


@traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a bronze, silver, or gold discount."""
    discounts = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discounts.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)


tools_dict = {
    "get_product_price": get_product_price,
    "apply_discount": apply_discount,
}


## 6. Test the actual functions locally


In [5]:
price = get_product_price("laptop")
print(price)
print(apply_discount(price, "gold"))


1299.99
1000.99


In [6]:
type(get_product_price)

function

## 7. Write manual function declarations

These dictionaries describe callable interfaces to Gemini. They do not execute the Python functions.


In [7]:
tools_for_llm = [
    {
        "name": "get_product_price",
        "description": "Look up the price of a product in the catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "product": {
                    "type": "string",
                    "description": "The product name",
                }
            },
            "required": ["product"],
        },
    },
    {
        "name": "apply_discount",
        "description": "Apply a bronze, silver, or gold discount.",
        "parameters": {
            "type": "object",
            "properties": {
                "price": {"type": "number"},
                "discount_tier": {"type": "string"},
            },
            "required": ["price", "discount_tier"],
        },
    },
]

print(json.dumps(tools_for_llm, indent=2))


[
  {
    "name": "get_product_price",
    "description": "Look up the price of a product in the catalog.",
    "parameters": {
      "type": "object",
      "properties": {
        "product": {
          "type": "string",
          "description": "The product name"
        }
      },
      "required": [
        "product"
      ]
    }
  },
  {
    "name": "apply_discount",
    "description": "Apply a bronze, silver, or gold discount.",
    "parameters": {
      "type": "object",
      "properties": {
        "price": {
          "type": "number"
        },
        "discount_tier": {
          "type": "string"
        }
      },
      "required": [
        "price",
        "discount_tier"
      ]
    }
  }
]


## 8. Define the system instruction


In [8]:
SYSTEM_INSTRUCTION = (
    "You are a helpful shopping assistant.\n"
    "You have access to a product catalog tool and a discount tool.\n\n "

    "STRICT RULES — you must follow these exactly:\n"
    
    "1. NEVER guess or assume any product price. "
    "You MUST call get_product_price first to get the real price.\n"
    
    "2. Only call apply_discount AFTER you have received "
    "a price from get_product_price. Pass the exact price "
    "returned by get_product_price — do NOT pass a made-up number.\n"
    
    "3. NEVER calculate discounts yourself using math. "
    "Always use the apply_discount tool.\n"
    
    "4. If the user does not specify a discount tier, "
    "ask them which tier to use — do NOT assume one."
)


## 9. Trace-friendly serialization

Gemini returns provider-specific objects. These helpers create readable copies for LangSmith without modifying what Gemini receives.


In [21]:
def serialize_gemini_part(part) -> str:
    """Convert a Gemini Part into readable text for LangSmith."""

    # Ordinary text response
    if getattr(part, "text", None):
        return part.text

    # Function call requested by Gemini
    function_call = getattr(part, "function_call", None,)

    if function_call is not None:
        function_name = function_call.name

        function_args = dict(
            function_call.args or {}
        )

        return (
            "FUNCTION CALL\n"
            f"Name: {function_name}\n"
            f"Arguments: "
            f"{json.dumps(function_args, indent=2)}"
        )

    # Result returned by the Python function
    function_response = getattr(part, "function_response", None,)

    if function_response is not None:
        function_name = function_response.name
        response_data = function_response.response

        return (
            "FUNCTION RESPONSE\n"
            f"Name: {function_name}\n"
            f"Response: "
            f"{json.dumps(response_data, indent=2)}"
        )

    # Fallback for any other Gemini Part type
    if hasattr(part, "model_dump"):
        serialized_part = part.model_dump(
            mode="json",
            exclude_none=True,
        )

        return json.dumps(
            serialized_part,
            indent=2,
        )

    return str(part)


def serialize_gemini_message(message) -> dict:
    """Convert Gemini Content into a readable LangSmith message."""

    parts = message.parts or []

    # Gemini calls model-generated messages "model".
    # LangSmith calls model-generated messages "assistant".
    if message.role == "model":
        trace_role = "assistant"
    else:
        trace_role = message.role

    content_parts = [
        serialize_gemini_part(part)
        for part in parts
    ]

    return {
        "role": trace_role,
        "content": "\n".join(content_parts),
    }

def process_gemini_trace_inputs(inputs: dict,) -> dict:
    """Control how Gemini inputs appear in LangSmith."""

    # Add the system instruction as a visible system message
    # in the trace-only representation.

    trace_messages = [
        {
            "role": "system",
            "content": inputs["system_instruction"],
        }
    ]

    # Add the complete accumulated conversation history.

    trace_messages.extend(
        serialize_gemini_message(message)
        for message in inputs["messages"]
    )

    return {
        "model": inputs["model"],
        "messages": trace_messages,
        "tool_schemas": inputs["tools"],
    }


def process_gemini_trace_outputs(output,) -> dict:
    """Control how Gemini outputs appear in LangSmith."""

    if (
        output is None
        or not getattr(output, "candidates", None)
    ):
        return {
            "messages": [],
            "error": "Gemini returned no candidates",
        }

    ai_message = output.candidates[0].content
    formatted_message = serialize_gemini_message(ai_message)

    return {
        "messages": [formatted_message],
        "model_version": getattr(output, "model_version", None,),
        "response_id": getattr(output, "response_id", None, ),
    }


## 10. Define the traced raw Gemini call


In [13]:
@traceable(
    name="Gemini Generate Content",
    run_type="llm",
    process_inputs = process_gemini_trace_inputs,
    process_outputs = process_gemini_trace_outputs,
)
def gemini_chat_traced(messages, system_instruction, tools, model):
    gemini_tools = types.Tool(function_declarations=tools)
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=[gemini_tools],
    )
    return client.models.generate_content(
        model=model,
        contents=messages,
        config=config,
    )


## 11. Build the initial native Gemini history


In [14]:
question = "What is the price of a laptop after applying a gold discount?"

messages = [
    types.Content(
        role="user",
        parts=[types.Part(text=question)],
    )
]

print(messages)


[Content(
  parts=[
    Part(
      text='What is the price of a laptop after applying a gold discount?'
    ),
  ],
  role='user'
)]


## 12. Optional first provider call

> **This cell makes one Gemini request.** It lets you inspect the native `FunctionCall` before running the whole loop.


In [15]:
first_response = gemini_chat_traced(
    messages=messages,
    system_instruction=SYSTEM_INSTRUCTION,
    tools=tools_for_llm,
    model=MODEL,
)

print(first_response.function_calls)


[FunctionCall(
  args={
    'product': 'laptop'
  },
  id='call_142925',
  name='get_product_price'
)]


## 13. Define the complete raw loop


In [16]:
@traceable(name="Gemini Agent Loop")
def run_agent(question: str):
    messages = [
        types.Content(role="user", parts=[types.Part(text=question)])
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"--- Iteration {iteration} ---")

        response = gemini_chat_traced(
            messages=messages,
            system_instruction=SYSTEM_INSTRUCTION,
            tools=tools_for_llm,
            model=MODEL,
        )

        ai_message = response.candidates[0].content
        tool_calls = response.function_calls or []

        if not tool_calls:
            return response.text, messages + [ai_message]

        tool_call = tool_calls[0]
        tool_name = tool_call.name
        tool_args = dict(tool_call.args or {})

        print("Tool selected:", tool_name)
        print("Arguments:", tool_args)

        selected_tool = tools_dict.get(tool_name)
        if selected_tool is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        observation = selected_tool(**tool_args)
        print("Observation:", observation)

        messages.append(ai_message)
        function_response = types.Part.from_function_response(
            name=tool_name,
            response={"result": observation},
        )
        messages.append(
            types.Content(role="user", parts=[function_response])
        )

    raise RuntimeError("Maximum iterations reached")


## 14. Run the raw Gemini agent

> **This cell normally makes three Gemini requests.**


In [17]:
final_answer, final_messages = run_agent(question)
print("Final answer:", final_answer)


--- Iteration 1 ---
Tool selected: get_product_price
Arguments: {'product': 'laptop'}
Observation: 1299.99
--- Iteration 2 ---
Tool selected: apply_discount
Arguments: {'price': 1299.99, 'discount_tier': 'gold'}
Observation: 1000.99
--- Iteration 3 ---
Final answer: The price of a laptop after applying a gold discount is $1000.99.


## 15. Inspect native history and serialized history


In [23]:
for index, message in enumerate(final_messages):
    print("=" * 70)
    print("Native role:", message.role)
    # print("Message:", message)
    print("Serialized copy:", serialize_gemini_message(message))


Native role: user
Serialized copy: {'role': 'user', 'content': 'What is the price of a laptop after applying a gold discount?'}
Native role: model
Serialized copy: {'role': 'assistant', 'content': 'FUNCTION CALL\nName: get_product_price\nArguments: {\n  "product": "laptop"\n}'}
Native role: user
Serialized copy: {'role': 'user', 'content': 'FUNCTION RESPONSE\nName: get_product_price\nResponse: {\n  "result": 1299.99\n}'}
Native role: model
Serialized copy: {'role': 'assistant', 'content': 'FUNCTION CALL\nName: apply_discount\nArguments: {\n  "price": 1299.99,\n  "discount_tier": "gold"\n}'}
Native role: user
Serialized copy: {'role': 'user', 'content': 'FUNCTION RESPONSE\nName: apply_discount\nResponse: {\n  "result": 1000.99\n}'}
Native role: model
Serialized copy: {'role': 'assistant', 'content': 'The price of a laptop after applying a gold discount is $1000.99.'}


## 16. Key takeaways

- Schema and implementation are separate.
- Gemini chooses a function name and named arguments.
- Python executes the actual function with `**tool_args`.
- Gemini history uses native `Content` and `Part` objects.
- A function result is represented by `from_function_response()`.
- Custom serialization is for trace readability only.
- Raw SDK code provides control but is provider-specific and longer.

## Practice exercises

- Add an enum to `discount_tier`.
- Add a fourth product.
- Print the history length per iteration.
- Compare raw Gemini fields with LangChain’s normalized `tool_calls`.
